# Importações

In [62]:
# Instalar dependências necessárias
import subprocess
import sys

# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "click"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spacy"])


In [ ]:
import os
import pandas as pd
import re
import spacy

# Configurações

In [ ]:
# Altere o paht conforme o caminho do seu computador, caso queira rodar o código localmente.
caminho_fake = 'C:/Users/samuel/OneDrive/Área de Trabalho/Fake.br-Corpus-master/full_texts/fake'
caminho_true = 'C:/Users/samuel/OneDrive/Área de Trabalho/Fake.br-Corpus-master/full_texts/true'

nlp = spacy.load('pt_core_news_sm')

# Funções

In [ ]:
def carregar_textos_com_titulo(caminho_pasta, classe_label):
    """
    Carrega e estrutura os textos de uma pasta específica, separando o título do corpo da notícia.

    Argumentos:
        caminho_pasta (str): Caminho do diretório contendo os arquivos .txt.
        classe_label (int): Rótulo da classe a ser atribuída aos textos (ex: 1 para Fake, 0 para True).

    Lógica:
        1. Itera sobre todos os arquivos com extensão .txt na pasta fornecida.
        2. Lê o conteúdo completo do arquivo.
        3. Tenta separar o título do corpo procurando por uma quebra de linha (\n).
        4. Caso não haja quebra de linha (bloco único), usa uma expressão regular (Regex) para 
           extrair a primeira frase (até o primeiro ponto, exclamação ou interrogação) como título.
        5. Trata a exceção de textos com apenas uma frase, duplicando-a para evitar falhas matemáticas.
        6. Agrupa tudo em um dicionário.

    Retorna:
        list: Uma lista de dicionários, onde cada dicionário representa uma notícia com as chaves: 
              'id_arquivo', 'titulo', 'corpo_noticia', 'texto_original' e 'classe'.
    """
    dados = []
    for nome_arquivo in os.listdir(caminho_pasta):
        if nome_arquivo.endswith('.txt'):
            caminho_completo = os.path.join(caminho_pasta, nome_arquivo)
            with open(caminho_completo, 'r', encoding='utf-8') as f:
                # Lê o arquivo inteiro de uma vez
                texto_completo = f.read().strip()
                
                # 1ª Tentativa: Separa por quebra de linha (\n)
                if '\n' in texto_completo:
                    partes = texto_completo.split('\n', 1)
                    titulo = partes[0].strip()
                    corpo = partes[1].strip()
                else:
                    # Se for um bloco contínuo, extrai a primeira frase como título.
                    # O regex abaixo corta o texto no primeiro ponto, exclamação ou interrogação.
                    partes = re.split(r'(?<=[.!?])\s+', texto_completo, maxsplit=1)
                    titulo = partes[0].strip() if len(partes) > 0 else ""
                    corpo = partes[1].strip() if len(partes) > 1 else ""
                
                # Se o texto for minúsculo e só tiver uma frase, duplica para não quebrar a matemática
                if not corpo:
                    corpo = titulo
                    
                dados.append({
                    'id_arquivo': nome_arquivo,
                    'titulo': titulo,
                    'corpo_noticia': corpo,
                    'texto_original': texto_completo,
                    'classe': classe_label
                })
    return dados
def limpar_texto_bert(texto):
    """
    Realiza uma limpeza estrutural leve no texto, ideal para modelos baseados em Transformers.

    Argumentos:
        texto (str): A string contendo o texto original da notícia.

    Lógica:
        1. Utiliza expressões regulares (Regex) para encontrar e remover hiperlinks (HTTP/WWW).
        2. Substitui múltiplas quebras de linha ou espaços duplicados por um único espaço em branco.
        3. Preserva a capitalização (maiúsculas/minúsculas), pontuação e stopwords, pois são vitais 
           para o contexto semântico do BERT.

    Retorna:
        str: O texto limpo de ruídos de formatação, pronto para o tokenizador WordPiece.
    """
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto
def calcular_pct_maiusculas(texto):
    """
    Calcula a proporção de letras maiúsculas em relação ao tamanho total do documento.

    Argumentos:
        texto (str): A string contendo o texto a ser analisado.

    Lógica:
        1. Verifica se o texto é vazio para evitar erro de divisão por zero.
        2. Conta o número total de caracteres em caixa alta (maiúsculas) na string.
        3. Divide essa contagem pelo número total de caracteres do texto.

    Retorna:
        float: Um valor decimal representando a porcentagem de letras maiúsculas no texto.
    """
    if len(texto) == 0: return 0
    qtd_maiusculas = sum(1 for c in texto if c.isupper())
    return qtd_maiusculas / len(texto)
def limpar_texto_ml_classico(texto):
    """
    Aplica uma limpeza linguística profunda, otimizada para algoritmos de ML Clássico (BoW/TF-IDF).

    Argumentos:
        texto (str): A string contendo o texto original da notícia.

    Lógica:
        1. Remove URLs e endereços de e-mail usando expressões regulares.
        2. Converte todo o texto para caracteres minúsculos (lowercase).
        3. Passa o texto pelo pipeline do spaCy para realizar tokenização.
        4. Itera sobre os tokens e descarta stopwords (palavras comuns sem valor semântico isolado), 
           pontuações, espaços vazios e números isolados.
        5. Extrai o lemma (forma canônica) de cada palavra restante para reduzir a dimensionalidade.

    Retorna:
        str: Uma string composta apenas pelos lemmas das palavras relevantes, separados por espaços.
    """
    # 1. Remove URLs e e-mails via Regex
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'\S+@\S+', '', texto)
    
    # 2. Converte para minúsculas
    texto = texto.lower()
    
    # 3. Processamento com spaCy (Tokenização e Lematização)
    doc = nlp(texto)
    
    tokens_limpos = []
    for token in doc:
        # Filtra stopwords, pontuação, espaços vazios e números isolados
        if not token.is_stop and not token.is_punct and not token.is_space and not token.like_num:
            tokens_limpos.append(token.lemma_)
            
    # 4. Reconstrói o texto
    return " ".join(tokens_limpos)

# Leitura dos Dados

# 1. Carrega os dados (1 para Fake, 0 para True)


In [ ]:
print("Carregando notícias falsas...")
dados_fake = carregar_textos_com_titulo(caminho_fake, classe_label=1)

Carregando notícias falsas...


In [67]:
print("Carregando notícias verdadeiras...")
dados_true = carregar_textos_com_titulo(caminho_true, classe_label=0)

Carregando notícias verdadeiras...


In [68]:
# 2. Cria o DataFrame e embaralha (shuffle) os dados
df = pd.DataFrame(dados_fake + dados_true)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total de notícias carregadas: {len(df)}")

Total de notícias carregadas: 7200


In [69]:
df[df['id_arquivo'] == '1.txt']

,id_arquivo,titulo,corpo_noticia,texto_original,classe
4079,1.txt,Kátia Abreu diz que vai colocar sua expulsão e...,A senadora Kátia Abreu (sem partido-TO) disse ...,Kátia Abreu diz que vai colocar sua expulsão e...,1
5882,1.txt,﻿O Podemos decidiu expulsar o deputado federa...,"O partido, que no passado chegou a cogitar lan...",﻿O Podemos decidiu expulsar o deputado federa...,0


In [70]:
df[['id_arquivo', 'titulo', 'corpo_noticia', 'texto_original', 'classe']].head(5)

,id_arquivo,titulo,corpo_noticia,texto_original,classe
0,546.txt,"Chorando as pitangas: Bonner posta frase de ""d...",O âncora do JN finalmente desabafou nas redes ...,"Chorando as pitangas: Bonner posta frase de ""d...",1
1,3278.txt,"Na Bahia, adolescente de 14 anos prevê a próp...",Uma jovem de 14 anos foi encontrada morta no ú...,"Na Bahia, adolescente de 14 anos prevê a próp...",1
2,1422.txt,"Segunda-feira, 12 de março.",Bom dia! Aqui estão os principais assuntos par...,"Segunda-feira, 12 de março. Bom dia! Aqui estã...",0
3,2158.txt,"A insanidade de Lula: ""Se me prenderem, eu vi...","Se me deixarem solto, viro presidente de novo...","A insanidade de Lula: ""Se me prenderem, eu vi...",1
4,3286.txt,"Criança apanha de mulher ""petista"" só porque u...","No último dia 17, uma eleitora do PT não cons...","Criança apanha de mulher ""petista"" só porque u...",1


# 2. Pré Processamento e Criação de Features

## Criação de Features

In [71]:
df['pct_maiusculas'] = df['texto_original'].apply(calcular_pct_maiusculas)

# Contagem de pontuação expressiva (! e ?)
df['qtd_pontuacao_expressiva'] = df['texto_original'].apply(lambda x: len(re.findall(r'[!?]', x)))

# Comprimento do texto em palavras
df['comprimento_texto'] = df['texto_original'].apply(lambda x: len(x.split()))

df['len_titulo'] = df['titulo'].apply(lambda x: len(x.split()))
df['len_corpo'] = df['corpo_noticia'].apply(lambda x: len(x.split()))

# Razão entre título e corpo (substitui 0 por 1 no denominador para evitar divisão por zero)
df['razao_titulo_corpo'] = df['len_titulo'] / df['len_corpo'].replace(0, 1)

df.head()

,id_arquivo,titulo,corpo_noticia,texto_original,classe,pct_maiusculas,qtd_pontuacao_expressiva,comprimento_texto,len_titulo,len_corpo,razao_titulo_corpo
0,546.txt,"Chorando as pitangas: Bonner posta frase de ""d...",O âncora do JN finalmente desabafou nas redes ...,"Chorando as pitangas: Bonner posta frase de ""d...",1,0.030274,2,177,12,165,0.072727
1,3278.txt,"Na Bahia, adolescente de 14 anos prevê a próp...",Uma jovem de 14 anos foi encontrada morta no ú...,"Na Bahia, adolescente de 14 anos prevê a próp...",1,0.040165,2,177,15,162,0.092593
2,1422.txt,"Segunda-feira, 12 de março.",Bom dia! Aqui estão os principais assuntos par...,"Segunda-feira, 12 de março. Bom dia! Aqui estã...",0,0.028472,1,632,4,628,0.006369
3,2158.txt,"A insanidade de Lula: ""Se me prenderem, eu vi...","Se me deixarem solto, viro presidente de novo...","A insanidade de Lula: ""Se me prenderem, eu vi...",1,0.037827,0,178,10,168,0.059524
4,3286.txt,"Criança apanha de mulher ""petista"" só porque u...","No último dia 17, uma eleitora do PT não cons...","Criança apanha de mulher ""petista"" só porque u...",1,0.022092,0,263,12,251,0.047809


## Limpeza

In [ ]:
print("Iniciando limpeza profunda ...")
df['texto_limpo_ml'] = df['texto_original'].apply(limpar_texto_ml_classico)
df['texto_limpo_bert'] = df['texto_original'].apply(limpar_texto_bert)
print("Limpeza  concluída!")

Iniciando limpeza profunda (Isso pode demorar alguns minutos)...
Limpeza clássica concluída!


In [73]:
df.to_csv('dataset_pre_processado.csv', index=False)